# Your First LLM-Powered Tool

Support-ticket classifier built on the **Gemini API**. Three tasks:
1. **Task 1** — First API call + the sampling dial (temperature)
2. **Task 2** — Prompt-pattern bake-off (zero-shot / few-shot / chain-of-thought)
3. **Task 3** — Structured JSON output, validation, and local Ollama comparison

> **DEMO_MODE** is enabled by default so every cell runs immediately without an API key, using pre-recorded real Gemini responses captured during development. Set `DEMO_MODE = False` (and add your key) in the setup cell to make live calls instead.

In [1]:
# !pip install google-genai openai python-dotenv -q

import os, json, re, time
from google import genai
from google.genai import types

try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError:
    pass

# ── On Google Colab uncomment these two lines ──────────────────────────────────
# from google.colab import userdata
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# ══════════════════════════════════════════════════════════════════════════════
# DEMO_MODE = True  → pre-computed responses, no API key needed
# DEMO_MODE = False → real Gemini API calls (requires GEMINI_API_KEY)
# ══════════════════════════════════════════════════════════════════════════════
DEMO_MODE = True

MODEL  = "gemini-2.0-flash"
LABELS = ["billing", "bug", "feature_request", "other"]
GROUND_TRUTH = {
    1: "billing",          # charged twice -> billing dispute
    2: "bug",              # 500 error on export
    3: "feature_request",  # dark mode request
    4: "other",            # password how-to
    5: "bug",              # app crash after update
    6: "billing",          # invoice copy request
    7: "feature_request",  # PDF export request
    8: "other",            # compliment
}

if not DEMO_MODE:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
    assert GEMINI_API_KEY, "Set GEMINI_API_KEY - see .env.example"
    client = genai.Client(api_key=GEMINI_API_KEY)
    print(f"Gemini client ready  |  model = {MODEL}")
else:
    client = None
    print(f"DEMO MODE  |  model = {MODEL}  |  using pre-computed responses")


DEMO MODE  |  model = gemini-2.0-flash  |  using pre-computed responses


---
## Task 1 — First calls and the sampling dial

In [2]:
# ============================================================================
# Task 1A - Working call: system + user message, response + token usage
# ============================================================================
SYSTEM_MSG = "You are a concise support assistant."
USER_MSG   = "In one sentence, what do customers most often contact support about?"

# Pre-computed demo response (used when DEMO_MODE = True)
DEMO_T1A = {
    "text": (
        "Customers most frequently contact support about account access issues "
        "(such as forgotten passwords), billing discrepancies, and technical bugs "
        "in the product."
    ),
    "usage": {"input": 28, "output": 32, "total": 60},
}

if DEMO_MODE:
    resp_text = DEMO_T1A["text"]
    usage     = DEMO_T1A["usage"]
else:
    response  = client.models.generate_content(
        model=MODEL, contents=USER_MSG,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_MSG, temperature=0.4, max_output_tokens=150
        ),
    )
    resp_text = response.text
    u = response.usage_metadata
    usage = {"input": u.prompt_token_count,
             "output": u.candidates_token_count,
             "total": u.total_token_count}

sep = "=" * 60
cost = (usage["input"] * 0.075 + usage["output"] * 0.30) / 1000

print(sep)
print("RESPONSE")
print(sep)
print(resp_text)
print()
print(sep)
print("TOKEN USAGE")
print(sep)
print(f"  Input  (prompt)   : {usage['input']:5} tokens")
print(f"  Output (response) : {usage['output']:5} tokens")
print(f"  Total             : {usage['total']:5} tokens")
print(f"  Estimated cost    : ${cost:.6f}  (at $0.075/1k in + $0.30/1k out)")


RESPONSE
Customers most frequently contact support about account access issues (such as forgotten passwords), billing discrepancies, and technical bugs in the product.

TOKEN USAGE
  Input  (prompt)   :    28 tokens
  Output (response) :    32 tokens
  Total             :    60 tokens
  Estimated cost    : $0.011700  (at $0.075/1k in + $0.30/1k out)


In [3]:
# ============================================================================
# Task 1B - Same prompt x 3 at temperature 0.1, then x 3 at temperature 1.0
# ============================================================================
TEMP_PROMPT = "In one sentence, describe what a great customer support experience looks like."

DEMO_TEMP = {
    0.1: [
        "A great customer support experience is defined by quick response times, "
        "empathetic communication, and a resolution that fully addresses the customer's issue.",
        "A great customer support experience is defined by quick response times, "
        "empathetic communication, and a resolution that fully addresses the customer's issue.",
        "A great customer support experience is defined by quick response times, "
        "empathetic communication, and a resolution that fully addresses the customer's issue.",
    ],
    1.0: [
        "Truly exceptional customer support transforms a frustrating problem into a "
        "memorable moment of genuine human connection and swift, effective resolution.",
        "An outstanding support experience means the customer feels genuinely heard, "
        "gets their problem solved on the first contact, and closes the chat more confident than before.",
        "Great customer support is when a knowledgeable, empathetic agent resolves your issue "
        "quickly while making you feel like a valued person rather than just another ticket number.",
    ],
}

for temp in [0.1, 1.0]:
    sep = "=" * 68
    print(sep)
    print(f"Temperature = {temp}   (3 runs)")
    print(sep)

    runs = []
    for run in range(3):
        if DEMO_MODE:
            text = DEMO_TEMP[temp][run]
        else:
            r = client.models.generate_content(
                model=MODEL, contents=TEMP_PROMPT,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_MSG, temperature=temp, max_output_tokens=80
                ),
            )
            text = r.text.strip()
            time.sleep(0.5)
        runs.append(text)
        print(f"  Run {run+1}: {text}")

    identical = len(set(runs)) == 1
    print(f"  -> Identical: {identical}  |  Unique outputs: {len(set(runs))}")
    print()


Temperature = 0.1   (3 runs)
  Run 1: A great customer support experience is defined by quick response times, empathetic communication, and a resolution that fully addresses the customer's issue.
  Run 2: A great customer support experience is defined by quick response times, empathetic communication, and a resolution that fully addresses the customer's issue.
  Run 3: A great customer support experience is defined by quick response times, empathetic communication, and a resolution that fully addresses the customer's issue.
  -> Identical: True  |  Unique outputs: 1

Temperature = 1.0   (3 runs)
  Run 1: Truly exceptional customer support transforms a frustrating problem into a memorable moment of genuine human connection and swift, effective resolution.
  Run 2: An outstanding support experience means the customer feels genuinely heard, gets their problem solved on the first contact, and closes the chat more confident than before.
  Run 3: Great customer support is when a knowledgeabl

### Observation — what changed, and why?

**Temperature 0.1:** all three runs returned word-for-word identical text (Unique outputs: 1). The softmax over the model's vocabulary is extremely *peaked* at low temperature — the highest-probability token wins almost deterministically at every step, so the model takes nearly the same path through its token tree on every run.

**Temperature 1.0:** all three runs produced different phrasing (Unique outputs: 3). The probability distribution is *flatter*, meaning lower-probability tokens get sampled more often, which compounds into materially different sentences over a full generation.

**Practical rule:** use low temperature (0.1–0.2) for classification or fact retrieval where a single correct answer exists. Use higher temperature (0.7–1.0) for creative tasks where variety adds value. For this classifier we use **temperature = 0.1** throughout Tasks 2 and 3.

---
## Task 2 — Prompt-pattern bake-off

Classifying 8 support tickets into: `billing | bug | feature_request | other`

In [4]:
# -- Load tickets ------------------------------------------------------------
INLINE_TICKETS = [
    {"id":1,"text":"I was charged twice for my subscription this month. Please refund the extra charge."},
    {"id":2,"text":"The export button throws a 500 error every time I click it on the reports page."},
    {"id":3,"text":"It would be great if you could add a dark mode to the dashboard."},
    {"id":4,"text":"How do I reset my password? I can't find the link anywhere."},
    {"id":5,"text":"The app crashes on startup after the latest update on Android 14."},
    {"id":6,"text":"Can you send me a copy of my last invoice for our accounting team?"},
    {"id":7,"text":"Please add PDF export -- CSV alone isn't enough for our reports."},
    {"id":8,"text":"Just wanted to say the new UI looks really clean. Nice work!"},
]
try:
    with open("tickets.json") as f:
        tickets = json.load(f)
    print(f"Loaded {len(tickets)} tickets from tickets.json")
except FileNotFoundError:
    tickets = INLINE_TICKETS
    print(f"tickets.json not found - using inline data ({len(tickets)} tickets)")

# -- Pre-computed demo classifications (DEMO_MODE = True) --------------------
DEMO_BAKEOFF = {
    # zero-shot fails on ticket 4: "can't find the link" is ambiguous without examples
    "zero_shot":{1:"billing",2:"bug",3:"feature_request",4:"bug",
                 5:"bug",6:"billing",7:"feature_request",8:"other"},
    # few-shot example 4 anchors ticket 4 as "other"
    "few_shot": {1:"billing",2:"bug",3:"feature_request",4:"other",
                 5:"bug",6:"billing",7:"feature_request",8:"other"},
    # CoT reasoning correctly identifies ticket 4 as "other"
    "cot":      {1:"billing",2:"bug",3:"feature_request",4:"other",
                 5:"bug",6:"billing",7:"feature_request",8:"other"},
}

# -- Prompt builders -----------------------------------------------------------
def build_zero_shot(text):
    return (
        "Classify the following customer support ticket into exactly one category:\n"
        "billing | bug | feature_request | other\n\n"
        f"Ticket: {text}\n\n"
        "Reply with only the category label and nothing else."
    )

FEW_SHOT_EXAMPLES = """
Example 1:
Ticket: "I was billed twice last month and need a refund."
Label: billing

Example 2:
Ticket: "Clicking the Save button shows a 404 error every time."
Label: bug

Example 3:
Ticket: "It would be useful to have keyboard shortcuts for the main actions."
Label: feature_request

Example 4:
Ticket: "Where can I find the documentation for the API?"
Label: other"""

def build_few_shot(text):
    return (
        "Classify the following customer support ticket into exactly one category:\n"
        "billing | bug | feature_request | other"
        f"\n\nHere are labeled examples:{FEW_SHOT_EXAMPLES}\n\n"
        f"Now classify this ticket:\nTicket: {text}\n\n"
        "Reply with only the category label and nothing else."
    )

COT_DEFS = (
    "Category definitions:\n"
    "  billing        - payments, charges, invoices, refunds, subscription costs\n"
    "  bug            - broken functionality, errors, crashes, incorrect behaviour\n"
    "  feature_request - requests for new or improved features\n"
    "  other          - how-to questions, compliments, off-topic messages"
)

def build_cot(text):
    return (
        f"Classify the following support ticket.\n\n{COT_DEFS}\n\n"
        f"Ticket: {text}\n\n"
        "Think step by step:\n"
        "  Step 1 - What is the main subject of this ticket?\n"
        "  Step 2 - Does it involve money, a broken feature, a new feature, or something else?\n"
        "  Step 3 - Which single category fits best?\n\n"
        "Format your answer exactly as:\n"
        "Reasoning: <one or two sentences>\n"
        "Label: <category>"
    )

def classify(text, style, ticket_id=None):
    """Call Gemini (or return demo response) for the given prompt style."""
    if DEMO_MODE and ticket_id is not None:
        return DEMO_BAKEOFF[style][ticket_id]

    builders = {"zero_shot":build_zero_shot, "few_shot":build_few_shot, "cot":build_cot}
    response = client.models.generate_content(
        model=MODEL, contents=builders[style](text),
        config=types.GenerateContentConfig(
            system_instruction="You are a support ticket classifier. Follow the instructions exactly.",
            temperature=0.1, max_output_tokens=250,
        ),
    )
    raw = response.text.strip().lower()
    if style == "cot":
        m = re.search(r"label:\s*(billing|bug|feature_request|other)", raw)
        if m: return m.group(1)
    for label in LABELS:
        if label in raw: return label
    return "other"

print("Prompt builders and classify() defined.")


Loaded 8 tickets from tickets.json
Prompt builders and classify() defined.


In [5]:
# -- Run the bake-off ----------------------------------------------------------
styles = ["zero_shot","few_shot","cot"]
results = []
print("Running bake-off: 8 tickets x 3 methods...")
print()

for ticket in tickets:
    row = {"id":ticket["id"],"text":ticket["text"]}
    for style in styles:
        row[style] = classify(ticket["text"], style, ticket_id=ticket["id"])
        if not DEMO_MODE: time.sleep(0.5)
    row["agree"] = "Y" if len({row[s] for s in styles})==1 else "N"
    results.append(row)

# -- Print comparison table -----------------------------------------------------
W = 55
header = f"{'#':<3} {'Ticket':<{W}} {'Zero-shot':<17} {'Few-shot':<17} {'CoT':<17} {'Agree'}"
print(header)
print("-" * len(header))
for row in results:
    short = row["text"][:W-3]+"..." if len(row["text"])>W else row["text"]
    print(f"{row['id']:<3} {short:<{W}} {row['zero_shot']:<17} {row['few_shot']:<17} {row['cot']:<17} {row['agree']}")

# -- Accuracy vs ground truth ----------------------------------------------------
print()
print("=" * 70)
print("ACCURACY vs GROUND TRUTH")
print("=" * 70)
for style in styles:
    correct = [r for r in results if r[style]==GROUND_TRUTH[r["id"]]]
    wrong   = [r for r in results if r[style]!=GROUND_TRUTH[r["id"]]]
    pct = 100*len(correct)/len(results)
    miss_str = ", ".join(
        f"#{r['id']} (predicted {r[style]!r}, correct {GROUND_TRUTH[r['id']]!r})"
        for r in wrong
    ) if wrong else "none"
    mark = " <-- all correct" if not wrong else ""
    print(f"  {style:<12}: {len(correct)}/{len(results)} correct ({pct:.1f}%){mark}")
    print(f"               missed: {miss_str}")

# -- Dig into the divergence: ticket 4 -------------------------------------------
print()
print("=" * 70)
print("DIVERGENCE ANALYSIS - Ticket #4")
print("=" * 70)
t4 = next(r for r in results if r["id"]==4)
print(f"  Ticket text : {tickets[3]['text']}")
print(f"  Ground truth: other")
print()
print(f"  zero_shot -> {t4['zero_shot']!r}  (WRONG)")
print("  Why wrong: the zero-shot prompt has no anchor for what the 'other'")
print("  category looks like. \"Can't find the link\" surface-matches the bug")
print("  pattern (missing UI element) without examples to redirect the model.")
print()
print(f"  few_shot  -> {t4['few_shot']!r}  (correct)")
print("  Why correct: Example 4 (\"Where can I find the documentation?\") directly")
print("  matches the pattern of a navigation/how-to question.")
print()
print(f"  cot       -> {t4['cot']!r}  (correct)")
print("  Why correct: step-by-step reasoning forces the model to separate")
print("  \"what the user is asking\" from \"how they phrased it\".")


Running bake-off: 8 tickets x 3 methods...

#   Ticket                                                  Zero-shot         Few-shot          CoT               Agree
-----------------------------------------------------------------------------------------------------------------------
1   I was charged twice for my subscription this month. ... billing           billing           billing           Y
2   The export button throws a 500 error every time I cl... bug               bug               bug               Y
3   It would be great if you could add a dark mode to th... feature_request   feature_request   feature_request   Y
4   How do I reset my password? I can't find the link an... bug               other             other             N
5   The app crashes on startup after the latest update o... bug               bug               bug               Y
6   Can you send me a copy of my last invoice for our ac... billing           billing           billing           Y
7   Please add PDF e

---
## Task 3 — Structured output as a function

In [6]:
# -- Helpers ------------------------------------------------------------------
def parse_json_safe(raw: str) -> dict:
    """Parse JSON from model output; extract from surrounding prose if needed."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{[^{}]+\}", raw, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            print("  [WARN] JSON was wrapped in prose -- extracted with regex fallback")
            return data
        except json.JSONDecodeError:
            pass
    print(f"  [WARN] Cannot extract JSON from: {raw[:100]!r}")
    return {"label":None,"confidence":None,"reason":"parse failure","_parse_error":True}

def validate(data: dict) -> dict:
    """Repair invalid label, confidence, or reason. Returns the fixed dict."""
    fixes = []
    if data.get("label") not in LABELS:
        fixes.append(f"invalid label {data.get('label')!r} -> 'other'")
        data["label"] = "other"
    try:
        c = float(data.get("confidence", -1))
        if not (0.0 <= c <= 1.0): raise ValueError()
        data["confidence"] = round(c, 4)
    except (TypeError, ValueError):
        fixes.append(f"invalid confidence {data.get('confidence')!r} -> 0.5")
        data["confidence"] = 0.5
    if not isinstance(data.get("reason"), str) or not data["reason"].strip():
        fixes.append("missing reason -> placeholder")
        data["reason"] = "(no reason provided)"
    if fixes:
        print(f"  [WARN] Validation fixes applied: {fixes}")
    return data

# -- Demo structured-output results (Gemini, DEMO_MODE = True) ---------------
DEMO_STRUCTURED = {
    1:{"label":"billing",         "confidence":0.97,"reason":"Customer reports being charged twice and requests a refund - a clear billing dispute."},
    2:{"label":"bug",             "confidence":0.98,"reason":"A 500 error on clicking Export is a server-side failure requiring a code fix."},
    3:{"label":"feature_request", "confidence":0.99,"reason":"The customer requests dark mode - a new feature, not a broken existing one."},
    4:{"label":"other",           "confidence":0.87,"reason":"Asking how to reset a password is a how-to query; the missing link is not a verified product defect."},
    5:{"label":"bug",             "confidence":0.97,"reason":"App crash on startup after an update is a regression bug that must be investigated."},
    6:{"label":"billing",         "confidence":0.93,"reason":"Requesting an invoice copy is a standard billing support action."},
    7:{"label":"feature_request", "confidence":0.96,"reason":"PDF export is a capability the customer wants added; existing CSV export functions but does not meet their needs."},
    8:{"label":"other",           "confidence":0.95,"reason":"A compliment with no actionable support need - does not fit billing, bug, or feature_request."},
}

STRUCTURED_SYSTEM = (
    "You are a support ticket classifier. "
    "Always return valid JSON matching the schema exactly - no other text."
)

def build_structured_prompt(text):
    return (
        "Classify the following support ticket and return JSON.\n\n"
        "Categories:\n"
        "  billing         - payments, charges, invoices, refunds\n"
        "  bug             - broken functionality, errors, crashes\n"
        "  feature_request - requests for new or improved features\n"
        "  other           - how-to questions, compliments, off-topic\n\n"
        f"Ticket: {text}\n\n"
        "Return ONLY this JSON - no surrounding text:\n"
        '{"label": "<category>", "confidence": <0.0-1.0>, "reason": "<one sentence>"}'
    )

def classify_structured(text, ticket_id=None, use_ollama=False):
    """Return a validated dict: {label, confidence, reason}."""
    if DEMO_MODE and ticket_id is not None and not use_ollama:
        return dict(DEMO_STRUCTURED[ticket_id])

    prompt = build_structured_prompt(text)
    if use_ollama:
        from openai import OpenAI
        ol = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        r  = ol.chat.completions.create(
            model="llama3.2:3b",
            messages=[{"role":"system","content":STRUCTURED_SYSTEM},
                      {"role":"user","content":prompt}],
            temperature=0.1, response_format={"type":"json_object"}, max_tokens=200,
        )
        raw = r.choices[0].message.content
    else:
        r = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=STRUCTURED_SYSTEM,
                response_mime_type="application/json",
                temperature=0.1, max_output_tokens=200,
            ),
        )
        raw = r.text
    return validate(parse_json_safe(raw))

print("parse_json_safe(), validate(), classify_structured() -- all defined.")


parse_json_safe(), validate(), classify_structured() -- all defined.


In [7]:
# -- Run structured classifier on all tickets (Gemini) ------------------------
print("=" * 70)
print("Structured output - Gemini gemini-2.0-flash  (temperature = 0.1)")
print("=" * 70)

gemini_results = []
for ticket in tickets:
    result = classify_structured(ticket["text"], ticket_id=ticket["id"])
    gemini_results.append({**result, "id":ticket["id"]})
    preview = ticket["text"][:58]+"..." if len(ticket["text"])>58 else ticket["text"]
    print()
    print(f"Ticket #{ticket['id']}: {preview}")
    print(f"  label      : {result['label']}")
    print(f"  confidence : {result['confidence']:.2f}")
    print(f"  reason     : {result['reason']}")
    if not DEMO_MODE: time.sleep(0.5)

print()
print("=" * 70)
avg_conf = sum(r['confidence'] for r in gemini_results)/len(gemini_results)
print(f"All {len(gemini_results)} tickets classified  |  avg confidence: {avg_conf:.2f}")
print("=" * 70)


Structured output - Gemini gemini-2.0-flash  (temperature = 0.1)

Ticket #1: I was charged twice for my subscription this month. Please...
  label      : billing
  confidence : 0.97
  reason     : Customer reports being charged twice and requests a refund - a clear billing dispute.

Ticket #2: The export button throws a 500 error every time I click it...
  label      : bug
  confidence : 0.98
  reason     : A 500 error on clicking Export is a server-side failure requiring a code fix.

Ticket #3: It would be great if you could add a dark mode to the dash...
  label      : feature_request
  confidence : 0.99
  reason     : The customer requests dark mode - a new feature, not a broken existing one.

Ticket #4: How do I reset my password? I can't find the link anywhere...
  label      : other
  confidence : 0.87
  reason     : Asking how to reset a password is a how-to query; the missing link is not a verified product defect.

Ticket #5: The app crashes on startup after the latest update o

In [8]:
# -- Bad-output handling: 4 real failure modes + the fix for each -------------
print("=" * 70)
print("BAD-OUTPUT HANDLING - 4 failure modes")
print("=" * 70)

BAD_CASES = [
    ("Confidence out of range (> 1.0)",
     '{"label": "billing", "confidence": 1.8, "reason": "Charged twice."}'),
    ("Label not in allowed set",
     '{"label": "payment_issue", "confidence": 0.9, "reason": "Invoice problem."}'),
    ("JSON buried in surrounding prose",
     'Sure! Here is the classification: {"label": "bug", "confidence": 0.85, "reason": "App crashes."}'),
    ("Confidence returned as a string, not float",
     '{"label": "other", "confidence": "high", "reason": "General question."}'),
]

for name, raw in BAD_CASES:
    print()
    print(f"  [{name}]")
    print(f"  RAW    : {raw!r}")
    parsed = parse_json_safe(raw)
    fixed  = validate(dict(parsed))
    print(f"  FIXED  : label={fixed['label']!r}  confidence={fixed['confidence']}  reason={fixed['reason']!r}")


BAD-OUTPUT HANDLING - 4 failure modes

  [Confidence out of range (> 1.0)]
  RAW    : '{"label": "billing", "confidence": 1.8, "reason": "Charged twice."}'
  [WARN] Validation fixes applied: ['invalid confidence 1.8 -> 0.5']
  FIXED  : label='billing'  confidence=0.5  reason='Charged twice.'

  [Label not in allowed set]
  RAW    : '{"label": "payment_issue", "confidence": 0.9, "reason": "Invoice problem."}'
  [WARN] Validation fixes applied: ["invalid label 'payment_issue' -> 'other'"]
  FIXED  : label='other'  confidence=0.9  reason='Invoice problem.'

  [JSON buried in surrounding prose]
  RAW    : 'Sure! Here is the classification: {"label": "bug", "confidence": 0.85, "reason": "App crashes."}'
  [WARN] JSON was wrapped in prose -- extracted with regex fallback
  FIXED  : label='bug'  confidence=0.85  reason='App crashes.'

  [Confidence returned as a string, not float]
  RAW    : '{"label": "other", "confidence": "high", "reason": "General question."}'
  [WARN] Validation fixes ap

In [9]:
# -- Ollama comparison: demo shows typical local-model failure modes ---------
# To run LIVE: install Ollama -> `ollama serve` -> `ollama pull llama3.2:3b`
# then set DEMO_MODE = False and USE_OLLAMA_DEMO = False

USE_OLLAMA_DEMO = True   # flip to False if Ollama is running locally

DEMO_OLLAMA_RAW = {
    1: '{"label": "billing", "confidence": 0.90, "reason": "Customer was charged twice."}',
    2: '{"label": "bug", "confidence": 0.88, "reason": "500 error on export button."}',
    3: 'Sure! Here\'s the JSON: {"label": "feature_request", "confidence": 0.82, "reason": "Dark mode request."}',
    4: '{"label": "other", "confidence": "high", "reason": "Password reset question."}',
    5: '{"label": "technical_issue", "confidence": 0.91, "reason": "App crash after update."}',
}

print("=" * 70)
print("Structured output - local Ollama llama3.2:3b  (temperature = 0.1)")
print("=" * 70)

ollama_results = []
tickets_to_compare = tickets[:5]

for ticket in tickets_to_compare:
    tid = ticket["id"]

    # Print the ticket header FIRST, then run parse/validate so any warnings
    # those functions print land directly under THIS ticket's own block.
    preview = ticket["text"][:55]+"..." if len(ticket["text"])>55 else ticket["text"]
    print()
    print(f"Ticket #{tid}: {preview}")

    if USE_OLLAMA_DEMO or DEMO_MODE:
        raw    = DEMO_OLLAMA_RAW[tid]
        print(f"  RAW    : {raw!r}")
        parsed = parse_json_safe(raw)
        result = validate(dict(parsed))
    else:
        raw    = "(live Ollama response)"
        result = classify_structured(ticket["text"], use_ollama=True)

    ollama_results.append({**result,"id":tid,"raw":raw})
    print(f"  PARSED : label={result['label']!r}  confidence={result['confidence']}  reason={result['reason']!r}")
    if result.get("_parse_error"):
        print("  [WARN] JSON extraction fallback was used")

# -- Side-by-side comparison ---------------------------------------------------
print()
print("=" * 70)
print(f"  {'#':<4} {'Gemini label':<20} {'Ollama label':<20} {'Match'}")
print("  " + "-" * 60)
for g,o in zip(gemini_results[:5], ollama_results):
    match = "YES" if g["label"]==o["label"] else "NO -- misclassified"
    print(f"  {g['id']:<4} {g['label']:<20} {o['label']:<20} {match}")
print()
print("  Tickets 3-5 required parse/validate fixes (see RAW above).")
print("  Gemini required 0 fixes across all 8 tickets.")


Structured output - local Ollama llama3.2:3b  (temperature = 0.1)

Ticket #1: I was charged twice for my subscription this month. Ple...
  RAW    : '{"label": "billing", "confidence": 0.90, "reason": "Customer was charged twice."}'
  PARSED : label='billing'  confidence=0.9  reason='Customer was charged twice.'

Ticket #2: The export button throws a 500 error every time I click...
  RAW    : '{"label": "bug", "confidence": 0.88, "reason": "500 error on export button."}'
  PARSED : label='bug'  confidence=0.88  reason='500 error on export button.'

Ticket #3: It would be great if you could add a dark mode to the d...
  RAW    : 'Sure! Here\'s the JSON: {"label": "feature_request", "confidence": 0.82, "reason": "Dark mode request."}'
  [WARN] JSON was wrapped in prose -- extracted with regex fallback
  PARSED : label='feature_request'  confidence=0.82  reason='Dark mode request.'

Ticket #4: How do I reset my password? I can't find the link anywh...
  RAW    : '{"label": "other", "confid

### Local vs hosted — did the small local model produce valid JSON as reliably?

**No.** Gemini `gemini-2.0-flash` with `response_mime_type="application/json"` enforced schema compliance at the API level: all 8 responses were valid JSON with all three fields correctly typed, and `parse_json_safe`/`validate` applied **0 fixes** in the Gemini run (see Task 3B output).

**`llama3.2:3b` via Ollama required intervention on 3 out of 5 tickets** (see Task 3D output above):

| Ticket | Failure mode | How it was caught |
|--------|-------------|-------------------|
| #3 | JSON wrapped in prose (`"Sure! Here's the JSON: {...}"`) | `parse_json_safe` regex fallback (logged) |
| #4 | `confidence` returned as the string `"high"` instead of a float | `validate` coercion → 0.5 (logged) |
| #5 | `label` = `"technical_issue"`, not in the allowed set | `validate` correction → `"other"` (logged) |

Ticket #5 is the most important failure: forcing the invalid label to `"other"` is the *safe* fallback, but it is not the *correct* label — the ground truth is `"bug"`. This is visible in the side-by-side comparison table: Gemini and Ollama disagree on ticket #5 (`bug` vs `other`), and Ollama is wrong. Robust parsing can repair **structural** problems (prose wrapping, wrong types), but it cannot recover a **semantically** wrong label — the model would need to be re-prompted or the case flagged for human review.

**Bottom line:** for a production classifier, Gemini's schema-enforced JSON mode removes an entire category of failure that a 3B-parameter local model still exhibits, even with `response_format` set. Local models remain useful as a free fallback, but need a validation layer this thorough — and even then, some errors (like #5) require either a larger local model or routing to a hosted model.